# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You will learn how to identify record sets, fields, and columns by their `@id` and load the data for analysis and visualization.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`. All references to records, record sets, fields, and columns are via their `@id` as per Croissant best practices.

In [ ]:
# List all record sets and their fields by @id
record_sets = []
field_ids = dict()

print("Available record sets (by @id):")
for rs in metadata.record_sets:
    print(f"  - {rs.id} ({rs.name})")
    record_sets.append(rs.id)
    # For each record set, list its field @id and names
    print("    Fields:")
    field_ids[rs.id] = []
    for field in rs.fields:
        print(f"      - {field.id} ({field.name})")
        field_ids[rs.id].append(field.id)
    print()

> _Note: If the printout above is empty, the dataset's Croissant schema currently does not declare any top-level record sets or record sets may exist in the data files listed under `metadata.distributions`. In that case, consult the Croissant schema or documentation for the appropriate record set `@id`._

## 3. Data Extraction
Load data from **each available record set** into a pandas DataFrame for further analysis.
All extractions use `@id` values for record sets and fields.

In [ ]:
# Load each available record set (using @id), if any
dfs = {}
for rs_id in record_sets:
    print(f"Extracting records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"  Loaded DataFrame with {len(df)} rows, columns: {df.columns.tolist()}")
        # Preview first few rows
        display(df.head())
    else:
        print(f"  No records found for record set {rs_id}.")
if not dfs:
    print("No tabular data could be extracted via mlcroissant record sets. Please check the dataset schema or provenance.")

> _To analyze and visualize data, select a specific record set and fields (by `@id`) from those loaded above. Example analysis below assumes at least one tabular record set was loaded._

## 4. Exploratory Data Analysis (EDA)
Apply processing steps such as filtering, normalization, and grouping on the selected DataFrame. **All columns referenced by their `@id`.**

In [ ]:
# Example: Choose record set and field @ids for analysis.
if dfs:
    # Pick the first available record set for example purposes
    rs_id = list(dfs.keys())[0]
    df = dfs[rs_id]

    print(f"Columns in {rs_id}: {df.columns.tolist()}")

    # Find a numeric field and a group field to use
    numeric_field = None
    group_field = None

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if numeric_field is None:
        print("No numeric field detected for EDA.")
    else:
        print(f"Using numeric field: {numeric_field}")

        # Example: Filter records where numeric_field > 10 (change threshold as appropriate)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with '{numeric_field}' > {threshold}.")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Group by a string/categorical field if possible
        if group_field and group_field in filtered_df.columns:
            print(f"Grouping by '{group_field}'...")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No data available for EDA. Please examine earlier cells for data extraction issues.")

## 5. Visualization
Visualize data distributions or relationships using matplotlib. Use field `@id` as axis labels.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data or numeric variable available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant-based dataset with the `mlcroissant` library.
- Identify record sets, fields, and columns by their `@id`.
- Extract data and perform simple EDA and visualization by referencing the Croissant schema.

Continue exploring the dataset by inspecting other record sets and customizing your analyses using additional field `@id`s and groupings.
